Morgane Bona-Pellissier (master 2 pluriTAL)

# TP 2 — Construire (et dépasser) son propre système de traduction statistique

**Fil rouge de ce TP :** en 1993, IBM Model 1 + un décodeur simple, c'était l'état de l'art. Vous allez le reconstruire vous-même, sur votre vrai corpus, puis mesurer exactement l'écart qui nous sépare de 2024/2025 (NLLB, LLM). Le nombre que vous allez obtenir à la fin est un vrai résultat expérimental — personne ne connaît sa valeur à l'avance pour votre corpus.

**Vous allez :**
1. aligner vos phrases (vous l'avez déjà fait — on réutilise l'outillage) ;
2. implémenter vous-même le E-step et le M-step de l'algorithme EM pour IBM Model 1 ;
3. en tirer un **lexique bilingue automatique** — un sous-produit réellement utilisé pour les langues peu dotées ;
4. construire un traducteur complet avec le canal bruité (modèle de traduction + modèle de langue + décodeur) ;
5. **évaluer** vos traductions (BLEU/sacreBLEU, COMET si disponible) et les comparer à NLLB-200 ou un LLM ;
6. discuter, avec vos propres exemples, de ce qui casse — et pourquoi.

Les cellules marquées **`# TODO`** sont à compléter. Le reste est fourni pour que vous puissiez vous concentrer sur l'algorithme, pas sur la plomberie.

## 0. Installation

In [ ]:
# INSTALL
# Importe les bibliothèques de base (le pip est commenté : tout est déjà dans le venv).
#!pip -q install sentence-transformers sacrebleu unbabel-comet nltk datasets
import re, math, random
from collections import defaultdict, Counter
import numpy as np
random.seed(0)

## 1. Charger votre corpus

Déposez vos fichiers `src.txt` (langue source) et `tgt.txt` (langue cible, une unité par ligne). Le code cherche ces fichiers ; à défaut, il utilise un petit corpus-jouet allemand → français (inventé, pour tester que le pipeline tourne — **remplacez-le par votre corpus dès que possible**).

In [ ]:
# CHARGEMENT
# Lit yi.txt et fr.txt bruts et les découpe en phrases avec wtpsplit -> phrases["yi"], phrases["fr"].
# wtpsplit est listé dans requirements.txt

import os
from wtpsplit import SaT

sat = SaT("sat-3l-sm")  # petit modèle, suffisant pour ce TP

def load_lines(path, toy):
    if os.path.exists(path):
        # encoding="utf-8-sig" permet de supprimer le caractère BOM
        with open(path, encoding="utf-8-sig") as f:
            return [l.strip() for l in f if l.strip()]
    print(f"[info] {path} introuvable — corpus-jouet utilisé.")
    return toy

# textes bruts
yi_lines = load_lines("yi.txt", [])
fr_lines = load_lines("fr.txt", [])

phrases = {}
# Les résultats seront gardés dans phrases["yi"] et phrases["fr"]
for name, lines in [("yi", yi_lines), ("fr", fr_lines)]:
    phrases[name] = [s.strip()
        # On saute les titres
        for l in lines if not l.startswith("#")
        # On supprime les * de mise en forme
        for s in sat.split(l.replace("*", ""))]

    print(f"{name}: {len(lines)} lignes -> {len(phrases[name])} phrases")

    # Afficher 3 phrasese par langue
    for s in phrases[name][:3]:
        print("   ", s)

yi: 487 lignes -> 982 phrases
    — דײַנע ציין זענען ביינערנע קראַטן.
    הינטער זיי, אין אַ קרישטאָלענער צעל, דײַנע געשמידטע ווערטער.
    געדענק די עצה פון אַן עלטערן :
fr: 497 lignes -> 907 phrases
    — Tes dents sont des grilles d’os.
    Derrière les barreaux, dans une geôle de cristal, tes mots enchaînés.
    Souviens-toi du conseil d’un aîné : les mots coupables, qui ont versé dans ta coupe des perles empoisonnées, laisse-les sortir en liberté.


## 2. Alignement des phrases — à vous de jouer

Si vos deux fichiers ne sont **pas déjà alignés ligne à ligne** (segmentation littéraire, nombre de phrases différent...), c'est ici qu'on le corrige — exactement comme dans le TP précédent. Réutilisez la méthode qui avait le mieux marché sur votre corpus (`bertalign`, ou votre réaligneur par embeddings fait main).

In [3]:
# ALIGNEMENT 1 : alignement des paragraphes
# Retrait de la mise en forme avec "*" + omission des titres (qui commencent par "#"")
yi_par = [l.replace("*", "") for l in yi_lines if not l.startswith("#")]
fr_par = [l.replace("*", "") for l in fr_lines if not l.startswith("#")]
yi_par = [p for p in yi_par if p]   # retire les séparateurs "***" devenus vides
fr_par = [p for p in fr_par if p]
print(len(yi_par), "paragraphes yiddish /", len(fr_par), "paragraphes français")

466 paragraphes yiddish / 474 paragraphes français


In [4]:
# ALIGNEMENT 2 : calcul des vecteurs LaBSE de chaque paragraphe
# LaBSE transforme chaque paragraphe en vecteurs
# -> permettra de comparer le yiddish et le français sans traduire

from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/LaBSE")

def embed(sents):
    # normalize_embeddings=True ramène chaque vecteur à la longueur 1
    # Les vecteurs doivent être normalisés pour obtenir leur similarité cosinus (en 3.)
    # show_progress_bar=True affiche une barre de progression
    return embedder.encode(sents, normalize_embeddings=True, show_progress_bar=True)

emb_yi = embed(yi_par)
emb_fr = embed(fr_par)
print(emb_yi.shape, emb_fr.shape)

# Résultats obtenus :
# (466, 768) (474, 768)

Batches: 100%|██████████| 15/15 [00:00<00:00, 16.56it/s]

(466, 768) (474, 768)


In [5]:
# ALIGNEMENT 3 : calcul de la matrice de similarité
# Calcul de la similarité cosinus pour comparer chaque paragraphe yiddish à chaque paragraphe français (sim[i][j])
sim = emb_yi @ emb_fr.T
print(sim.shape)

# la colonne française la plus proche pour chaque ligne yiddish
meilleur = sim.argmax(axis=1)
print("10 premiers paragraphes yiddish -> français :", meilleur[:10])
print("similarité moyenne du meilleur candidat :", sim.max(axis=1).mean().round(2))
print("écart médian de position |j - i| :", np.median(np.abs(meilleur - np.arange(len(meilleur)))))

# Résultats obtenus :
# (466, 474)
# - similarité moyenne du meilleur candidat : 0,76
# - écart médian de position |j - i| : 4.0 (10 paragraphes supplémentaires en français)

(466, 474)
10 premiers paragraphes yiddish -> français : [0 1 2 3 4 5 6 7 8 9]
similarité moyenne du meilleur candidat : 0.76
écart médian de position |j - i| : 4.0


In [6]:
# ALIGNEMENT 4 : meilleur chemin
# chaque paire formée ajoute sa similarité
# chaque paragraphe sauté, côté yiddish ou côté français, retire 0,3 de pénalité (penalite=0.3)
# L'algorithme peut sauter lorsque cela rapporte plus que cela ne coûte
# (p. ex. pour sauter un paragraphe en trop qui décalerait le reste de l'alignement)

def align_dp(sim, penalite=0.3):
    n, m = sim.shape
    dp = np.full((n + 1, m + 1), -np.inf)
    dp[0, 0] = 0
    back = {}
    for a in range(n + 1):
        for b in range(m + 1):
            if a == 0 and b == 0:
                continue
            best, choice = -np.inf, None
            # apparier le paragraphe yiddish a-1 avec le français b-1
            if a > 0 and b > 0 and dp[a-1, b-1] + sim[a-1, b-1] > best:
                best, choice = dp[a-1, b-1] + sim[a-1, b-1], (a-1, b-1, 'match')
            # sauter un paragraphe yiddish
            if a > 0 and dp[a-1, b] - penalite > best:
                best, choice = dp[a-1, b] - penalite, (a-1, b, 'skip_src')
            # sauter un paragraphe français
            if b > 0 and dp[a, b-1] - penalite > best:
                best, choice = dp[a, b-1] - penalite, (a, b-1, 'skip_tgt')
            dp[a, b] = best
            back[(a, b)] = choice
    # remonter le chemin depuis la dernière case
    path, a, b = [], n, m
    while (a, b) != (0, 0):
        prev = back[(a, b)]
        if prev[2] == 'match':
            path.append((prev[0], prev[1]))
        a, b = prev[0], prev[1]
    return list(reversed(path))

pairs = align_dp(sim)
print(len(pairs), "paires 1-1")
print("yiddish non appariés :", len(yi_par) - len(pairs),
      "| français non appariés :", len(fr_par) - len(pairs))

# Résultats obtenus :
# 460 paires 1-1
# yiddish non appariés : 6 | français non appariés : 14

460 paires 1-1
yiddish non appariés : 6 | français non appariés : 14


In [7]:
# ALIGNEMENT 5 : vérification des paires
import html
from IPython.display import display, HTML

# 1. Vue d'ensemble des scores
scores = [sim[i, j] for i, j in pairs]
print("score moyen :", np.mean(scores).round(2), "| minimum :", np.min(scores).round(2))
print("paires sous 0,5 :", sum(s < 0.5 for s in scores), "sur", len(pairs))

# 2. Quels paragraphes ont été sautés ?
yi_pris = {i for i, _ in pairs}
fr_pris = {j for _, j in pairs}
yi_saute = [i for i in range(len(yi_par)) if i not in yi_pris]
fr_saute = [j for j in range(len(fr_par)) if j not in fr_pris]
print("yiddish sautés :", yi_saute)
print("français sautés :", fr_saute)

# 3. Les 5 paires les moins sûres, côte à côte
for i, j in sorted(pairs, key=lambda p: sim[p])[:5]:
    print(f"\n--- yi[{i}] <-> fr[{j}]  (score {sim[i, j]:.2f})")
    display(HTML(f"<p dir='rtl' style='text-align:right'>{html.escape(yi_par[i][:200])}</p>"))
    print(fr_par[j][:200])


print("=== Français sautés ===")
for j in fr_saute:
    print(f"fr[{j}] : {fr_par[j][:120]}")

print()
print("=== Yiddish sautés ===")
for i in yi_saute:
    print(f"yi[{i}] :")
    display(HTML(f"<p dir='rtl' style='text-align:right'>{html.escape(yi_par[i][:120])}</p>"))

# Résultats obtenus :
# score moyen : 0.76 | minimum : 0.25
# paires sous 0,5 : 10 sur 460

score moyen : 0.76 | minimum : 0.25
paires sous 0,5 : 10 sur 460
yiddish sautés : [22, 52, 84, 142, 144, 461]
français sautés : [25, 59, 70, 77, 194, 198, 203, 208, 233, 284, 349, 353, 440, 465]

--- yi[18] <-> fr[18]  (score 0.25)


Vert de sombres sapins à travers la brume…

--- yi[326] <-> fr[331]  (score 0.28)


Ses paupières éclatent.

--- yi[20] <-> fr[20]  (score 0.28)


Vert de pierres moussues sous la pluie…

--- yi[111] <-> fr[112]  (score 0.34)


Pour quelle raison, mon Dieu, a-t-il été condamné à la lapidation, l’oratoire du Gaon ? Et pour quel crime a-t-il été puni par le feu, le petit arbre du portail ?

--- yi[19] <-> fr[19]  (score 0.37)


...

yi[52] :


yi[84] :


yi[142] :


yi[144] :


yi[461] :


In [8]:
# ALIGNEMENT 6 : écriture de src.txt (yiddish) et tgt.txt (français)
src_aligne = [yi_par[i] for i, j in pairs]
tgt_aligne = [fr_par[j] for i, j in pairs]

# une paire par ligne (= pas de retour à la ligne au sein d'un paragraphe)
assert all("\n" not in p for p in src_aligne + tgt_aligne)
assert len(src_aligne) == len(tgt_aligne)

with open("src.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(src_aligne) + "\n")
with open("tgt.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(tgt_aligne) + "\n")

# Reconstruction de src_sents / tgt_sents
src_sents, tgt_sents = src_aligne, tgt_aligne
print(len(src_aligne), "paires écrites dans src.txt et tgt.txt")

460 paires écrites dans src.txt et tgt.txt


In [ ]:
# RECHARGEMENT & JEU DE TEST
# Recharge src.txt et tgt.txt dans src_sents / tgt_sents et vérifie qu'ils sont alignés.
# toy seront utilisés comme fichiers src et tgt si les fichiers fournis ne pointent nulle part.

toy_src = [
    "das Haus ist groß", "das Buch ist alt", "die Katze schläft", "der Hund läuft",
    "das Kind liest das Buch", "die Frau kauft das Haus", "der Mann liest die Zeitung",
    "das Mädchen sieht die Katze", "der Hund sieht das Kind", "die Frau liest das Buch",
]
toy_tgt = [
    "la maison est grande", "le livre est vieux", "le chat dort", "le chien court",
    "l'enfant lit le livre", "la femme achète la maison", "l'homme lit le journal",
    "la fille voit le chat", "le chien voit l'enfant", "la femme lit le livre",
]

src_sents = load_lines("src.txt", toy_src)
tgt_sents = load_lines("tgt.txt", toy_tgt)

assert len(src_sents) == len(tgt_sents), "src et tgt doivent avoir le même nombre de lignes : alignez-les d'abord (section 2) !"
print(len(src_sents), "paires de phrases chargées.")


# Jeu de test mis à l'écart de l'entraînement
N_TEST = 30
random.seed(0)   # tirage reproductible

# candidats : paires de 5 à 30 mots dans les deux langues
# candidats = les numéros k des paires (yiddish, français) qui peuvent servir de test
candidats = [k for k in range(len(src_sents))               # k parcourt les numéros de toutes les paires
             if 5 <= len(src_sents[k].split()) <= 30        # phrase yiddish de 5 à 30 mots (split() compte les mots séparés par des espaces)
             and 5 <= len(tgt_sents[k].split()) <= 30]      # ET phrase française de 5 à 30 mots

test_idx = set(random.sample(candidats, N_TEST))

test_src = [src_sents[k] for k in sorted(test_idx)]
test_ref = [tgt_sents[k] for k in sorted(test_idx)]

# l'entraînement ne voit plus ces paires
src_sents = [s for k, s in enumerate(src_sents) if k not in test_idx]
tgt_sents = [s for k, s in enumerate(tgt_sents) if k not in test_idx]
print(len(src_sents), "paires d'entraînement /", len(test_src), "paires de test")

# Résultat obtenu :
# "460 paires de phrases chargées.
# 430 paires d'entraînement / 30 paires de test"

460 paires de phrases chargées.
430 paires d'entraînement / 30 paires de test


## 3. Tokenisation et vocabulaire

In [10]:
# TOKENISATION

def tokenize(s):
    # split() coupe uniquement aux espaces
    #return s.lower().split()

    s = s.lower()

    # points de suspension
    s = re.sub(r"\.{2,}", "…", s)

    # élisions
    s = re.sub(r"\b(l|d|j|t|m|s|n|c|qu|jusqu|lorsqu|puisqu)(['’])(?=\w)", r"\1\2 ", s)

    # ponctuation : de chaque côté de chaque signe
    s = re.sub(r"([.,;:!?…«»()\[\]\"“”„—–])", r" \1 ", s)
    return s.split()

src_tok = [tokenize(s) for s in src_sents]
tgt_tok = [tokenize(s) for s in tgt_sents]

src_vocab = sorted(set(w for s in src_tok for w in s))
tgt_vocab = sorted(set(w for s in tgt_tok for w in s)) + ["<NULL>"]

print(f"{len(src_vocab)} mots source, {len(tgt_vocab)} mots cible (NULL compris).")
print(src_tok[0], "<->", tgt_tok[0])

# Résultats obtenu après correction :
# "2700 mots source, 2799 mots cible (NULL compris)."

2618 mots source, 2721 mots cible (NULL compris).
['—', 'דײַנע', 'ציין', 'זענען', 'ביינערנע', 'קראַטן', '.', 'הינטער', 'זיי', ',', 'אין', 'אַ', 'קרישטאָלענער', 'צעל', ',', 'דײַנע', 'געשמידטע', 'ווערטער', '.', 'געדענק', 'די', 'עצה', 'פון', 'אַן', 'עלטערן', ':', 'די', 'שולדיקע', ',', 'וואָס', 'האָבן', 'אַרײַנגעטאָן', 'אין', 'דײַן', 'בעכער', 'פאַרסמטע', 'פּערל', '—', 'זיי', 'לאָז', 'אַרויס', 'אויף', 'דער', 'פרײַ', '.', 'ווי', 'אַ', 'דאַנק', 'פאַרן', 'גענאָד', ',', 'וועלן', 'זיי', 'בויען', 'דײַן', 'אייביקייט', ';', 'נאָר', 'יענע', ',', 'די', 'אומשולדיקע', ',', 'וואָס', 'טרילערן', 'פאַלש', 'ווי', 'סאָלאָווייען', 'איבער', 'אַ', 'קבר', '—', 'זיי', 'זאָלסטו', 'ניט', 'שאַנעווען', '.', 'הענג', 'זיי', ',', 'ווער', 'זייער', 'תּלין', '!', 'ווײַל', 'קוים', 'לאָזסטו', 'זיי', 'אַרויס', 'פון', 'דײַן', 'מויל', ',', 'צי', 'פון', 'דײַן', 'פּען', '—', 'ווערן', 'זיי', 'שדים', '.', 'די', 'שטערן', 'זאָלן', 'אַזוי', 'ניט', 'אַראָפּפאַלן', 'ווי', 'איך', 'זאָג', 'דעם', 'אמת', '!', '—'] <-> ['—', 'tes', 'dents', 

In [11]:
# Vérification de la tokenisation (français)
tokenize("Souviens-toi du conseil d’un aîné : les mots, l’enfant... qu’il vienne !")

['souviens-toi',
 'du',
 'conseil',
 'd’',
 'un',
 'aîné',
 ':',
 'les',
 'mots',
 ',',
 'l’',
 'enfant',
 '…',
 'qu’',
 'il',
 'vienne',
 '!']

In [12]:
# Vérification de la tokenisation (yiddish)
tokenize("ווײַל איך, ליבער מענטש, וועל שוין לאַנג ניט ציִען, וועל גיין באַגעגענען די צוויי מיידלעך... גיט אַ האַנט,")

['ווײַל',
 'איך',
 ',',
 'ליבער',
 'מענטש',
 ',',
 'וועל',
 'שוין',
 'לאַנג',
 'ניט',
 'ציִען',
 ',',
 'וועל',
 'גיין',
 'באַגעגענען',
 'די',
 'צוויי',
 'מיידלעך',
 '…',
 'גיט',
 'אַ',
 'האַנט',
 ',']

## Résultat obtenu (français) :
```
['souviens-toi', 'du', 'conseil', 'd’', 'un', 'aîné', ':', 'les', 'mots', ',', 'l’', 'enfant', '…', 'qu’', 'il', 'vienne', '!']
```

## Résultat obtenu (yiddish) :
```
['ווײַל', 'איך', ',', 'ליבער', 'מענטש', ',', 'וועל', 'שוין', 'לאַנג', 'ניט', 'ציִען', ',', 'וועל', 'גיין', 'באַגעגענען', 'די', 'צוויי', 'מיידלעך', '…', 'גיט', 'אַ', 'האַנט', ',']
```

In [13]:
from collections import Counter

for nom, tok, vocab in [("yiddish", src_tok, src_vocab), ("français", tgt_tok, tgt_vocab)]:
    avec_ponct = [w for w in vocab if w[-1] in ".,;:!?»…)"]
    print(f"{nom} : {len(vocab)} mots distincts, dont {len(avec_ponct)} se terminent par de la ponctuation")
    print("   exemples :", avec_ponct[:8])
    print("   plus fréquents :", Counter(w for s in tok for w in s).most_common(10))

yiddish : 2618 mots distincts, dont 7 se terminent par de la ponctuation
   exemples : ['!', ',', '.', ':', ';', '?', '…']
   plus fréquents : [(',', 856), ('.', 602), ('אַ', 512), ('די', 310), ('אין', 294), ('—', 292), ('און', 259), ('פון', 229), ('דער', 211), ('זיך', 155)]
français : 2721 mots distincts, dont 8 se terminent par de la ponctuation
   exemples : ['!', ',', '.', ':', ';', '?', '»', '…']
   plus fréquents : [(',', 932), ('.', 614), ('de', 495), ('la', 336), ('un', 273), ('le', 244), ('et', 221), ('à', 205), ('une', 191), ('les', 190)]


## Résultats obtenus avant correction :
```
yiddish : 3390 mots distincts, dont 1184 se terminent par de la ponctuation
   exemples : ['!', '!..', '..', ':', ';', '?', '?..', 'אַדלער,']
   plus fréquents : [('אַ', 541), ('די', 320), ('—', 305), ('אין', 304), ('און', 267), ('פון', 240), ('דער', 222), ('האָט', 160), ('מיט', 156), ('זיך', 154)]
français : 3626 mots distincts, dont 1268 se terminent par de la ponctuation
   exemples : ['!', '!…', ':', ';', '?', '?…', 'abattu,', 'abréger…']
   plus fréquents : [('de', 527), ('la', 351), ('le', 258), ('un', 239), ('et', 226), ('à', 206), ('les', 195), ('une', 166), ('dans', 163), ('en', 143)]
```

## Résultats obtenus après correction :
```
yiddish : 2700 mots distincts, dont 7 se terminent par de la ponctuation
   exemples : ['!', ',', '.', ':', ';', '?', '…']
   plus fréquents : [(',', 902), ('.', 640), ('אַ', 544), ('די', 321), ('—', 309), ('אין', 305), ('און', 270), ('פון', 241), ('דער', 223), ('זיך', 164)]
français : 2799 mots distincts, dont 8 se terminent par de la ponctuation
   exemples : ['!', ',', '.', ':', ';', '?', '»', '…']
   plus fréquents : [(',', 979), ('.', 649), ('de', 527), ('la', 352), ('un', 290), ('le', 258), ('et', 227), ('à', 213), ('une', 202), ('les', 195)]
```

## 4. IBM Model 1 : construire le modèle de traduction

On va y aller **une seule idée à la fois**. Toute la mécanique (les boucles, l'accumulation sur le corpus) est déjà écrite pour vous : vous allez compléter **une seule fonction**, qui répond à une question très concrète.

Pour un mot source $f$, et les mots cibles candidats d'une phrase (ex. : la, maison, NULL), comment répartir « 1 unité de certitude » entre ces candidats, sachant les $t(f\mid e)$ actuels ?

### Un exemple numérique avant le code

Reprenons l'exemple de la Leçon 4. Phrase source : *das Haus*. Mots cibles candidats : *la*, *maison*. Avec $t$ initialisé uniformément à $\frac{1}{3}$ partout :

$$\text{dénominateur} = t(\text{das}\mid \text{la}) + t(\text{das}\mid \text{maison}) = \tfrac{1}{3} + \tfrac{1}{3} = \tfrac{2}{3}$$

$$P(a = \text{la} \mid \text{das}) = \frac{t(\text{das}\mid\text{la})}{\text{dénominateur}} = \frac{1/3}{2/3} = 0{,}5 \qquad P(a = \text{maison} \mid \text{das}) = 0{,}5$$

C'est tout ce que votre fonction doit calculer : **une division**, répétée pour chaque candidat.

In [14]:
# Prends une paire de phrases : une phrase yiddish et sa traduction française
# Pour un mot yiddish f, on ne sait pas quel est le mot français correspondant.
# IBM Model 1 répartit une unité de certitude entre tous les mots français de la phrase
# plus un mot français e est susceptible de produire f, plus il reçoit de part

def posteriors_for_word(f, e_candidats, t):
    """
    f           : un mot source (ex. 'das')
    e_candidats : la liste des mots cible de la phrase, NULL compris (ex. ['<NULL>', 'la', 'maison'])
    t           : le dictionnaire courant des t(f|e)

    Retourne un dictionnaire {e: probabilité} qui doit sommer à 1.
    """
    denominateur = sum(t[(f, e)] for e in e_candidats)  # <- fourni, c'est la somme du bas de la formule

    probas = {}
    for e in e_candidats:
        probas[e] = t[(f, e)] / denominateur

    return probas

In [15]:
# Testez votre fonction ici avant de continuer : avec t uniforme, vous devez obtenir 0,5 / 0,5.
t_test = defaultdict(lambda: 1.0 / 3)
resultat = posteriors_for_word("das", ["la", "maison"], t_test)
print(resultat)
assert abs(sum(resultat.values()) - 1.0) < 1e-6, "Les probabilités doivent sommer à 1."
assert abs(resultat["la"] - 0.5) < 1e-6, "Avec t uniforme, on doit retrouver 0,5 / 0,5."
print("Test réussi !")

{'la': 0.5, 'maison': 0.5}
Test réussi !


**Si le test passe, vous avez fini la partie difficile.** Le reste — répéter ce calcul sur tout le corpus, puis ré-estimer $t(f\mid e)$ — est entièrement fourni ci-dessous ; lisez-le, il ne fait qu'appeler la fonction que vous venez d'écrire.

In [16]:
def e_step(src_tok, tgt_tok, t):
    """Parcourt tout le corpus et accumule les comptes fractionnaires, en
    appelant posteriors_for_word() pour chaque mot source rencontré."""
    count = defaultdict(float)
    total_e = defaultdict(float)
    for f_sent, e_sent_raw in zip(src_tok, tgt_tok):
        e_sent = ["<NULL>"] + e_sent_raw
        for f in f_sent:
            probas = posteriors_for_word(f, e_sent, t)
            for e, p in probas.items():
                count[(f, e)] += p
                total_e[e] += p
    return count, total_e

def m_step(count, total_e):
    """Ré-estime t(f|e) = compte(f,e) / total(e). Une paire jamais vue ensemble reste à 0."""
    new_t = defaultdict(float)
    for (f, e), c in count.items():
        new_t[(f, e)] = c / total_e[e]
    return new_t

## 5. Entraînement : faites tourner EM (Expectation-Optimisation)

In [17]:
# Initialisation : t(f|e) uniforme sur tout le vocabulaire source, pour un vrai entraînement
t = defaultdict(lambda: 1.0 / len(src_vocab))

N_ITER = 10
for it in range(N_ITER):
    count, total_e = e_step(src_tok, tgt_tok, t)
    t = m_step(count, total_e)

# petit test de cohérence : la somme des t(f|e) sur f, pour un e fixé, doit faire ~1
s = sum(t[(f, tgt_vocab[0])] for f in src_vocab)
print(f"Somme de contrôle pour e={tgt_vocab[0]!r} (doit être proche de 1) : {s:.3f}")

Somme de contrôle pour e='!' (doit être proche de 1) : 1.000


## 6. Sous-produit utile : extraire un lexique bilingue

Pour chaque mot cible, quel mot source lui est le plus probablement associé ? C'est exactement ce que font les dictionnaires bilingues construits automatiquement pour les langues peu dotées.

In [18]:
def top_translation(e, k=3):
    scores = [(t[(f, e)], f) for f in src_vocab]
    scores.sort(reverse=True)
    return scores[:k]

for e in tgt_vocab:
    if e == "<NULL>":
        continue
    best = top_translation(e)
    print(f"{e:12s} -> " + ", ".join(f"{f} ({p:.2f})" for p, f in best))

!            -> ! (0.92), — (0.08), , (0.00)
&            -> ראָם (0.14), פאַרטריפטער (0.14), והאחים (0.14)
,            -> , (0.77), אַ (0.05), . (0.04)
.            -> . (0.81), אַ (0.05), , (0.04)
1953         -> 1954 (0.48), 1953 (0.48), — (0.03)
1954         -> 1954 (0.48), 1953 (0.48), — (0.03)
:            -> : (0.99), און (0.00), דער (0.00)
;            -> ; (0.68), האָט (0.08), איר (0.03)
?            -> ? (0.99), איז (0.01), — (0.01)
a            -> האָט (0.89), . (0.03), דער (0.02)
a-t-elle     -> קלערן (0.15), ס'קינד (0.15), מירעלען (0.15)
a-t-il       -> קומט (0.17), שׂריפה (0.08), קלויז (0.08)
abandon      -> אויף (0.34), שטאָט (0.07), דער (0.03)
abandonnes   -> צום־ווידערזען (0.06), זײַ-געזונט (0.06), וועסט (0.06)
abattu       -> שוואַרץ (0.15), קוואַלן (0.15), צעשמעלצן (0.15)
abeille      -> בין (0.87), איז (0.07), ווילדע (0.02)
abeilles     -> בינען (0.64), פינגער (0.20), — (0.07)
abord        -> ערשט (0.53), צו (0.46), — (0.00)
abîme        -> אָפּגרונט (0.51), אַן (0

**Question :** certains mots cible pointent-ils de façon convaincante vers NULL (aucun mot source ne leur correspond vraiment) ? Que représentent ces cas concrètement ?

## 7. Modèle de langue (fourni)

Un bigramme avec lissage de Laplace, entraîné sur le **monolingue cible** uniquement — exactement l'esprit du canal bruité : $P(e)$ ne nécessite aucune donnée parallèle.

In [19]:
class BigramLM:
    def __init__(self, sentences):
        self.unigrams = Counter()
        self.bigrams = Counter()
        self.vocab = set()
        for s in sentences:
            toks = ["<s>"] + tokenize(s) + ["</s>"]
            self.vocab.update(toks)
            for w in toks:
                self.unigrams[w] += 1
            for w1, w2 in zip(toks, toks[1:]):
                self.bigrams[(w1, w2)] += 1
        self.V = len(self.vocab)

    def logprob(self, sentence_tokens):
        toks = ["<s>"] + sentence_tokens + ["</s>"]
        lp = 0.0
        for w1, w2 in zip(toks, toks[1:]):
            num = self.bigrams[(w1, w2)] + 1
            den = self.unigrams[w1] + self.V
            lp += math.log(num / den)
        return lp

lm = BigramLM(tgt_sents)
print("logP('le chat dort') =", lm.logprob(tokenize("le chat dort")))

logP('le chat dort') = -29.162057662882194


## 8. Le décodeur : combiner $P(f \mid e)$ et $P(e)$

On simplifie radicalement la recherche ($\arg\max_e$ sur **toutes** les phrases possibles est impraticable à la main) : pour chaque mot source, on ne garde que ses $k$ meilleures traductions candidates, on génère des combinaisons, et on les score. Toute la mécanique de recherche est fournie — vous n'écrivez que **le score d'une seule combinaison**, qui est une simple addition.

In [20]:
import itertools, random

def candidate_words(f, k=3):
    """Les k meilleures traductions candidates d'un mot source, d'après t(f|e). Fourni."""
    scores = [(t[(f, e)], e) for e in tgt_vocab if e != "<NULL>"]
    scores.sort(reverse=True)
    return [e for _, e in scores[:k]]

def decode_greedy(f_sent_tokens):
    """Traduction mot-à-mot gloutonne : pour chaque mot source, prend sa meilleure traduction. Fourni."""
    return " ".join(candidate_words(f, k=1)[0] for f in f_sent_tokens)

### À vous : noter une combinaison candidate

Pour une phrase source $(f_1, \dots, f_m)$ et une combinaison candidate $(e_1, \dots, e_m)$ (une traduction candidate pour chaque mot, dans l'ordre), le score du canal bruité est :

$$\text{score} = \underbrace{\sum_{j=1}^{m} \log t(f_j \mid e_j)}_{\text{modèle de traduction}} \;+\; \underbrace{\log P(e_1, \dots, e_m)}_{\text{modèle de langue}}$$

Le second terme, c'est exactement `lm.logprob(...)` de la section 7.

In [21]:
def score_combo(f_sent_tokens, combo):
    """
    f_sent_tokens : les mots source, ex. ['die', 'katze', 'läuft']
    combo         : une combinaison candidate de mots cible, même longueur, ex. ('le', 'chat', 'court')

    Retourne le score du canal bruité (un seul nombre, plus il est grand, mieux c'est).
    """
    score_traduction = sum(math.log(t[(f, e)] + 1e-12) for f, e in zip(f_sent_tokens, combo))
    # Indice : zip(f_sent_tokens, combo) donne les paires (f_j, e_j) dans l'ordre.
    # Indice 2 : utilisez math.log(t[(f, e)] + 1e-12) -- le +1e-12 évite log(0).

    score_langue = lm.logprob(list(combo))  # <- fourni

    return score_traduction + score_langue

Le reste — générer les combinaisons candidates et garder la meilleure — est fourni : il ne fait qu'appeler `score_combo()` autant de fois que nécessaire.

In [22]:
def decode_noisy_channel(f_sent_tokens, k=3, n_samples=200):
    """Génère des combinaisons candidates (produit cartésien si petit, sinon tirage aléatoire),
    et renvoie celle qui maximise score_combo(). Fourni."""
    cand_lists = [candidate_words(f, k=k) for f in f_sent_tokens]

    total_combos = 1
    for c in cand_lists:
        total_combos *= len(c)

    if total_combos <= n_samples:
        combos = itertools.product(*cand_lists)
    else:
        combos = (tuple(random.choice(c) for c in cand_lists) for _ in range(n_samples))

    best_score, best_combo = -1e18, None
    for combo in combos:
        s = score_combo(f_sent_tokens, combo)
        if s > best_score:
            best_score, best_combo = s, combo

    return " ".join(best_combo) if best_combo else ""

test_sent = tokenize("die Katze läuft")
print("Mot-à-mot :", decode_greedy(test_sent))
print("Canal bruité :", decode_noisy_channel(test_sent))

Mot-à-mot : … … …
Canal bruité : … ” …


**Question :** sur quels types de phrases le décodeur « canal bruité » fait-il mieux que le simple mot-à-mot ? Sur quels types ne fait-il pas mieux — et pourquoi, d'après ce que vous savez des limites de Model 1 ?

## 9. Traduire un jeu de test

In [23]:
# Séparez quelques phrases de test (idéalement non vues à l'entraînement, sur votre vrai corpus).
#test_src = src_sents[-3:]
#test_ref = tgt_sents[-3:]

system_hyps = [decode_noisy_channel(tokenize(s)) or decode_greedy(tokenize(s)) for s in test_src]

for s, h, r in zip(test_src, system_hyps, test_ref):
    print(f"SRC: {s}\nHYP: {h}\nREF: {r}\n")

SRC: געפונען, געפונען! אָט איז דער חלום פון מײַן חלום — —
HYP: pourtant , pourtant ! vois est cet rêve noir mon rêve – colonnes
REF: Trouvé, je l’ai trouvé ! Le voici, le rêve de mon rêve…

SRC: כ'האָב גאָרנישט געפרעגט, נאָר דערלאַנגט אַן אָקרײַעץ ברויט, בײַ די ראַנדן פאַרשימלט.
HYP: pensais “ ” , mais sauvages métamorphosée ” moisi , poète les bordée ” .
REF: Sans rien demander, je lui tends un quignon de pain, moisi sur les bords.

SRC: — מײַן נאָמען איז פעליציע פּאָזנאַנסקי. דער שרײַבער י. י. זינגער האָט מיך פאַראייביקט אין אַ ראָמאַן. אַמאָל האָב איך אויסגעזען אַנדערש. נאָר דאָס איז ניט וויכטיק.
HYP: — ma nom est œuvre poznanski bonjour éclair écrivain allé bonjour allé . “ eut avais … rouge humaine … . demi-siècle pensais j’ ressemblait béante . mais mousse est pas … .
REF: — Mon nom est Félicia Poznanski. L’écrivain Israël Joshua Singer m’a immortalisée dans un roman. Autrefois j’avais fière allure. Mais là n’est pas l’important.

SRC: דער זונפאַרגאַנג באַלײַכט אירע קנייטשן מיטן 

## 10. Évaluation — votre système vs. l'état de l'art

On compare trois traducteurs sur le **même jeu de test** : votre IBM Model 1 + canal bruité, une traduction mot-à-mot (baseline minimale), et un modèle moderne (NLLB-200 ou un LLM via prompt).

In [24]:
import sacrebleu

baseline_hyps = [decode_greedy(tokenize(s)) for s in test_src]

def show_bleu(name, hyps, refs):
    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    print(f"{name:25s} sacreBLEU = {bleu.score:.1f}")
    return bleu.score

show_bleu("Mot-à-mot (baseline)", baseline_hyps, test_ref)
show_bleu("IBM Model 1 + canal bruité", system_hyps, test_ref)

Mot-à-mot (baseline)      sacreBLEU = 2.8
IBM Model 1 + canal bruité sacreBLEU = 1.3


1.2793187991824777

### Comparaison avec un système moderne (fourni)

Adaptez le code de langue NLLB à votre paire de langues (voir la liste des codes FLORES-200). Si votre langue n'est pas couverte, essayez un LLM via prompt (section suivante) à la place.

In [25]:
# TRANSFORMERS
try:
    from transformers import pipeline
    # Codes de langue NLLB adaptés à notre paire : yiddish ("ydd_Hebr") -> français ("fra_Latn")
    translator = pipeline("translation", model="facebook/nllb-200-distilled-600M",
                           src_lang="ydd_Hebr", tgt_lang="fra_Latn")
    nllb_hyps = [translator(s)[0]["translation_text"] for s in test_src]
    for s, h in zip(test_src, nllb_hyps):
        print(f"SRC: {s}\nNLLB: {h}\n")
    show_bleu("NLLB-200 (600M)", nllb_hyps, test_ref)
except Exception as e:
    print("NLLB indisponible pour cette paire de langues :", e)
    print("-> utilisez un LLM via prompt (cellule suivante) comme point de comparaison.")

SRC: געפונען, געפונען! אָט איז דער חלום פון מײַן חלום — —
NLLB: Trouvez, trouvez ! Voilà le rêve de mon rêve  

SRC: כ'האָב גאָרנישט געפרעגט, נאָר דערלאַנגט אַן אָקרײַעץ ברויט, בײַ די ראַנדן פאַרשימלט.
NLLB: Je n'ai rien demandé, mais j'ai offert un pain d'ocre, à la randonnée.

SRC: — מײַן נאָמען איז פעליציע פּאָזנאַנסקי. דער שרײַבער י. י. זינגער האָט מיך פאַראייביקט אין אַ ראָמאַן. אַמאָל האָב איך אויסגעזען אַנדערש. נאָר דאָס איז ניט וויכטיק.
NLLB: Je m'appelle Félicie Poznansky. L'écrivain Y. E. Singer m'a conçu dans un roman. J'avais l'air différent une fois. Mais ce n'est pas important.

SRC: דער זונפאַרגאַנג באַלײַכט אירע קנייטשן מיטן ליכט פון גרינע גליווערעס. זי ווערט דינער, אײַנגעשרומפּענער און זעט אויס ווי אַ מומיע פון אַן אַלט-עגיפּטישער פּרינצעסין
NLLB: Le coucher du soleil éclaire ses filles avec la lumière de glaives vertes. Elle devient plus servante, plus éblouissante et ressemble à une mumie d'une princesse égyptienne ancienne

SRC: — מען דאַרף ניט קיין וואַסער, מען דאַ

# Traductions obtenues avec Transformers
SRC: <span dir="rtl">געפונען, געפונען! אָט איז דער חלום פון מײַן חלום — —</span>  
NLLB: Trouvez, trouvez ! Voilà le rêve de mon rêve  

SRC: <span dir="rtl">כ'האָב גאָרנישט געפרעגט, נאָר דערלאַנגט אַן אָקרײַעץ ברויט, בײַ די ראַנדן פאַרשימלט.</span>  
NLLB: Je n'ai rien demandé, mais j'ai offert un pain d'ocre, à la randonnée.

SRC: <span dir="rtl">— מײַן נאָמען איז פעליציע פּאָזנאַנסקי. דער שרײַבער י. י. זינגער האָט מיך פאַראייביקט אין אַ ראָמאַן. אַמאָל האָב איך אויסגעזען אַנדערש. נאָר דאָס איז ניט וויכטיק.</span>  
NLLB: Je m'appelle Félicie Poznansky. L'écrivain Y. E. Singer m'a conçu dans un roman. J'avais l'air différent une fois. Mais ce n'est pas important.

SRC: <span dir="rtl">דער זונפאַרגאַנג באַלײַכט אירע קנייטשן מיטן ליכט פון גרינע גליווערעס. זי ווערט דינער, אײַנגעשרומפּענער און זעט אויס ווי אַ מומיע פון אַן אַלט-עגיפּטישער פּרינצעסין</span>  
NLLB: Le coucher du soleil éclaire ses filles avec la lumière de glaives vertes. Elle devient plus servante, plus éblouissante et ressemble à une mumie d'une princesse égyptienne ancienne

SRC: <span dir="rtl">— מען דאַרף ניט קיין וואַסער, מען דאַרף ניט. אַ צוויק אונטערן האַרצן, אַ נאַרישקייט. וווּ האַלטן מיר ? יאָ, כ'וועל מאַכן בקיצור —</span>  
NLLB: Il n'y a pas besoin d'eau, il n'y a pas besoin. Un excès sous le cœur, une folie.

SRC: <span dir="rtl">און אפשר איז גאָר שולדיק זײַן פּאַליטרע, האַ ?</span>  
NLLB: Et peut-être que c'est sa peinture qui est vraiment coupable, hein ?

SRC: <span dir="rtl">זײַנע וואַסערדיקע אויגן ווערן אויסגעפּוקלט מער. גיסן זיך איבער אויף דער פּאַליטרע. אַ נאַסע פלעמלדיקע ראָזקייט באַדעקט זײַן פּנים.</span>  
NLLB: Ses yeux aquatiques s'ouvrent de plus en plus. Ils se jettent sur la palette. Une rosée flamboyante humide couvre son visage.

SRC: <span dir="rtl">— אַי, צאַלקע, זײַן אייפערזיכטיק אין אַ קוימען... גוט, כ'וועל דיר זאָגן, ס'איז געווען — — —</span>  
NLLB: Ah, bonheur, être éphémère dans un coin... Bon, je vais vous dire, c'était...

SRC: <span dir="rtl">ס'נעמט וואָלקענען, רעגענען, מיט וואַרעמע באָבישע טרערן. זיי, די טרערן, פאַרלעשן דעם שײַטער, פילן אָן די קאַנאַווע.</span>  
NLLB: Il pleut, il pleut, avec les larmes chaleureuses de Bobbie. Elles, les larmes, oublient la nuée, se sentent sans la toile.

SRC: <span dir="rtl">קיינער באַפעלט ניט מע זאָל אויפשטיין. די ליפּן, ווי אָפּגעהאַקטע פינגער, קאָנען זיך ניט באַוועגן...</span>  
NLLB: Personne ne commande de se lever. Les lèvres, comme des doigts coupés, ne peuvent pas bouger...

SRC: <span dir="rtl">ס'האָט זיך דערהערט אַ שטעפּערײַ פֿון אַ קוילנזאַרפֿער, וואָס האָט אויפגעהויבן פון זומפּ אַ וואָרצל ווי אַ נאָז פון אַ העלפאַנד.</span>  
NLLB: On entendit une clameur d'un conservateur qui relevait d'une soupe une racine comme un nez d'un serpent.

SRC: <span dir="rtl">אויף מאָרגן, אין דולער בלאָנדזשעניש, האָב איך ענדלעך דערטאַפּט דאָס בערגל. נאָר פון מיידל איז דאָרט פאַרבליבן נאָך ווינציקער ווי פון אַ חלום.</span>  
NLLB: Le lendemain matin, dans une blonde douce, j'ai finalement atteint la montagne. Seule une jeune fille y est restée pendant vingt ans, comme si c'était un rêve.

SRC: <span dir="rtl">בײַם ווערן אַ פּעלצל ביסטו פאַרבליבן אַ שעפעלע.</span>  
NLLB: En devenant une pellicule, vous restez une pellicule.

SRC: <span dir="rtl">אַ גײַסט בין איך געווען אין דײַנע פליגל. פון זיך אַליין פאַרשוווּנדן. זיבן שטראַליקע טויבן האָבן מיך געבראַכט אַהיים.</span>  
NLLB: J'étais un hôte dans tes ailes. Je me suis évanoui par moi-même. Sept étoiles brillantes m'ont ramené à la maison.

SRC: <span dir="rtl">ווען כ'בין געוואָרן פאַר דיר צו גרויס, האָסטו מיר אָפּגעגעבן דײַנע הענט — די צוויי אַרבל, צו פאַרלענגערן די פרײַנדשאַפט.</span>  
NLLB: Quand je suis devenu trop grand pour toi, tu m'as donné les deux mains pour prolonger l'amitié.

SRC: <span dir="rtl">צו זײַנע אויערן דערטראָגט זיך אַ מממו, אַ גרויליק בענקענדיקער מממו, אַ דונער מיט אָפּגעהאַקטע פליגל...</span>  
NLLB: À ses oreilles se présente une mamme, une mamme grossière, une donnée aux ailes démembrées...

SRC: <span dir="rtl">— ניין, ברודערקע, אַזאַ בוים דערקענט מען בײַ נאַכט. אַרום אים, זעסט, — אַ גאָלדענער שלייער.</span>  
NLLB: Non, ma sœur, on reconnaît un tel arbre pendant la nuit. Il est entouré d'une couche d'or.

SRC: <span dir="rtl">— ער וועט קלעקן אָפּצובראָטן אַ סטאַדע אָקסן, טרוקן ווי פּעך, — זעצט יאָאַכים די האַק אין בוים אַרײַן.</span>  
NLLB:  Il écrasera une stade d'oiseaux, secs comme la paille,  Joachim ramasse la hache dans l'arbre.

SRC: <span dir="rtl">אַ גראָע צעכוואַליעטע וואַנט, ווי אין דער וואַנט וואָלטן זיך געטרונקען וועלף, שווימט אַרויס הינטער די זײַלן פון רעגן.</span>  
NLLB: Un mur gris et déchiré, comme dans le mur, plonge à l'extérieur derrière les branches de la pluie.

SRC: <span dir="rtl">— אורשוליאַ, וויפל איז דער זייגער ?</span>  
NLLB:  Ursula, quelle est l'heure ?

SRC: <span dir="rtl">דער רעגן טוט אַ זעץ אין שויב מיט זײַנע בלײַענע פויסטן.</span>  
NLLB: La pluie fait une sieste dans la poussière avec ses plumes blanches.

SRC: <span dir="rtl">זינט איך האָב געפונען דעם ייִנגל אויפן דעמב, איז ער געוואָרן מײַן שוואַרצאַפּל.</span>  
NLLB: Depuis que j'ai trouvé le garçon, il est devenu mon chapeau.

SRC: <span dir="rtl">און דער מענטש האָט אַרײַנגעקוקט אינעווייניק און גאָרנישט ניט געזען, ווײַל זײַן פּנים איז באַקלעפּט געוואָרן פון שפּינוועבס בײַם בייגן זיך איבערן מויל פון ברונעם.</span>  
NLLB: L'homme entre dans la maison, il regarde et il ne voit rien. Il a les yeux pleins d'or et il est plein de tristesse.

SRC: <span dir="rtl">און דעמאָלט האָט ער פון בוזעם אויסגעפּלאָנטערט אַ שטריק, זי פאַרקניפּט אַרום אַ האָק, און ווי אַ קוימענקערער אין אַ קוימען זיך אַרונטערגעלאָזן אין ברונעם.</span>  
NLLB: Et puis il a planté un plan de bosse, elle s'est enroulée autour d'un crochet, et comme un marchand dans un couloir se tourne autour dans le brun.

SRC: <span dir="rtl">ערשט דאָרטן, מיט אַ קלאַפּנדיק האַרץ, און מיט אויגן ווי ווילדע מאָנבלומען, האָט ער אין ביידע הענט אַרײַנגענומען דעם אוצר און עס געהאַלטן אַנטקעגן.</span>  
NLLB: D'abord là, avec un cœur battant, et des yeux comme des colliers sauvages, il prit dans ses deux mains le trésor et le déposa.

SRC: <span dir="rtl">און ער האָט נאָך אַ מאָל אין ביידע הענט אַרײַנגענומען דעם שאַרבן און ווי אַ הונט אַ צעשלאָגענער דערלאַנגט אַ וואָיע :</span>  
NLLB: Jsus prend encore une fois le morceau de pain dans ses deux mains. Un chien qui croit que le chien est mu et il part en criant:

SRC: <span dir="rtl">און ער האָט דערפילט, אַז דער קאָפּ, וואָס ער טראָגט שוין אַזוי פיל יאָרן אויף די אַקסלען, איז ניט זײַנער.</span>  
NLLB: Il s'est rendu compte que la tête qu'il portait depuis tant d'années n'était pas la sienne.

SRC: <span dir="rtl">דער מלאך מיט די פאַרסאַזשעטע פליגל טראָגט אַוועק דאָס געסל — דעם שוואַרצן אָרון.</span>  
NLLB: L'ange aux ailes déployées enlève l'éphod de l'Aaron noir.

SRC: <span dir="rtl">— דיר גלוסט זיך איבערשנײַדן, האַ ?</span>  
NLLB: Vous vous trompez, hein ?

SRC: <span dir="rtl">אַצינד זעט ער די לבנה מיט זײַן גלעזערן אויג. אויף דעם אַנדערן הענגט אַ שלאָס. דער זילבערנער שליסל איז ניט מער אויף דער וועלט.</span>  
NLLB: Un jour, il voit la lune avec son œil cristallin. De l'autre côté, il y a un verrou. La clef d'argent n'est plus sur la terre.

## Scores obtenus avec Transformers
NLLB-200 (600M)           **sacreBLEU = 13.4**

In [26]:
# TORCH
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    # Codes de langue NLLB adaptés à notre paire : yiddish ("ydd_Hebr") -> français ("fra_Latn")
    nom = "facebook/nllb-200-distilled-600M"
    tokenizer = AutoTokenizer.from_pretrained(nom, src_lang="ydd_Hebr")
    model = AutoModelForSeq2SeqLM.from_pretrained(nom)
    appareil = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(appareil)

    def translator(s):
        entree = tokenizer(s, return_tensors="pt").to(appareil)
        sortie = model.generate(**entree,
                                forced_bos_token_id=tokenizer.convert_tokens_to_ids("fra_Latn"),
                                max_new_tokens=256)
        return tokenizer.decode(sortie[0], skip_special_tokens=True)

    nllb_hyps = [translator(s) for s in test_src]
    for s, h in zip(test_src, nllb_hyps):
        print(f"SRC: {s}\nNLLB: {h}\n")
    show_bleu("NLLB-200 (600M)", nllb_hyps, test_ref)
except Exception as e:
    print("NLLB indisponible pour cette paire de langues :", e)
    print("-> utilisez un LLM via prompt (cellule suivante) comme point de comparaison.")


SRC: געפונען, געפונען! אָט איז דער חלום פון מײַן חלום — —
NLLB: Trouver, trouver! Voilà le rêve de mon rêve

SRC: כ'האָב גאָרנישט געפרעגט, נאָר דערלאַנגט אַן אָקרײַעץ ברויט, בײַ די ראַנדן פאַרשימלט.
NLLB: Je n'ai rien demandé, mais j'ai offert un pain d'ocre, à la ronde.

SRC: — מײַן נאָמען איז פעליציע פּאָזנאַנסקי. דער שרײַבער י. י. זינגער האָט מיך פאַראייביקט אין אַ ראָמאַן. אַמאָל האָב איך אויסגעזען אַנדערש. נאָר דאָס איז ניט וויכטיק.
NLLB: Je m'appelle Félicie Poznansky. L'écrivain Y.E. Singer m'a fait un roman. J'ai déjà été différent. Mais ce n'est pas important.

SRC: דער זונפאַרגאַנג באַלײַכט אירע קנייטשן מיטן ליכט פון גרינע גליווערעס. זי ווערט דינער, אײַנגעשרומפּענער און זעט אויס ווי אַ מומיע פון אַן אַלט-עגיפּטישער פּרינצעסין
NLLB: Le soleil se couche et elle éclaire ses filles de la lumière des glissières vertes. Elle devient plus servante, plus éblouissante et ressemble à une mumie d'une princesse égyptienne ancienne

SRC: — מען דאַרף ניט קיין וואַסער, מען דאַרף ניט. אַ צוו

## Traductions obtenues avec Torch
SRC: <span dir="rtl">געפונען, געפונען! אָט איז דער חלום פון מײַן חלום — —</span>  
NLLB: Trouver, trouver! Voilà le rêve de mon rêve

SRC: <span dir="rtl">כ'האָב גאָרנישט געפרעגט, נאָר דערלאַנגט אַן אָקרײַעץ ברויט, בײַ די ראַנדן פאַרשימלט.</span>  
NLLB: Je n'ai rien demandé, mais j'ai offert un pain d'ocre, à la ronde.

SRC: <span dir="rtl">— מײַן נאָמען איז פעליציע פּאָזנאַנסקי. דער שרײַבער י. י. זינגער האָט מיך פאַראייביקט אין אַ ראָמאַן. אַמאָל האָב איך אויסגעזען אַנדערש. נאָר דאָס איז ניט וויכטיק.</span>  
NLLB: Je m'appelle Félicie Poznansky. L'écrivain Y.E. Singer m'a fait un roman. J'ai déjà été différent. Mais ce n'est pas important.

SRC: <span dir="rtl">דער זונפאַרגאַנג באַלײַכט אירע קנייטשן מיטן ליכט פון גרינע גליווערעס. זי ווערט דינער, אײַנגעשרומפּענער און זעט אויס ווי אַ מומיע פון אַן אַלט-עגיפּטישער פּרינצעסין</span>  
NLLB: Le soleil se couche et elle éclaire ses filles de la lumière des glissières vertes. Elle devient plus servante, plus éblouissante et ressemble à une mumie d'une princesse égyptienne ancienne

SRC: <span dir="rtl">— מען דאַרף ניט קיין וואַסער, מען דאַרף ניט. אַ צוויק אונטערן האַרצן, אַ נאַרישקייט. וווּ האַלטן מיר ? יאָ, כ'וועל מאַכן בקיצור —</span>  
NLLB: Il n'y a pas besoin d'eau, il n'y a pas besoin. Un excès sous le cœur, une folie. Où me tenir ? Oui, je vais faire un résumé.

SRC: <span dir="rtl">און אפשר איז גאָר שולדיק זײַן פּאַליטרע, האַ ?</span>  
NLLB: Et peut-être que c'est son palette qui est vraiment coupable, hein ?

SRC: <span dir="rtl">זײַנע וואַסערדיקע אויגן ווערן אויסגעפּוקלט מער. גיסן זיך איבער אויף דער פּאַליטרע. אַ נאַסע פלעמלדיקע ראָזקייט באַדעקט זײַן פּנים.</span>  
NLLB: Ses yeux aquatiques s'ouvrent plus fort. Ils se jettent sur la palette. Une rosée flamboyante humide couvre son visage.

SRC: <span dir="rtl">— אַי, צאַלקע, זײַן אייפערזיכטיק אין אַ קוימען... גוט, כ'וועל דיר זאָגן, ס'איז געווען — — —</span>  
NLLB: Ah, bonheur, être éphérisé dans une cuisine... Bon, je vais te dire, c'était...

SRC: <span dir="rtl">ס'נעמט וואָלקענען, רעגענען, מיט וואַרעמע באָבישע טרערן. זיי, די טרערן, פאַרלעשן דעם שײַטער, פילן אָן די קאַנאַווע.</span>  
NLLB: Il pleut, il pleut, avec des larmes chaleureuses de Bob. Elles, les larmes, oublient la terre, se sentent sans la toile.

SRC: <span dir="rtl">קיינער באַפעלט ניט מע זאָל אויפשטיין. די ליפּן, ווי אָפּגעהאַקטע פינגער, קאָנען זיך ניט באַוועגן...</span>  
NLLB: Personne ne nous ordonne de nous lever. Les lèvres, comme des doigts coupés, ne peuvent bouger...

SRC: <span dir="rtl">ס'האָט זיך דערהערט אַ שטעפּערײַ פֿון אַ קוילנזאַרפֿער, וואָס האָט אויפגעהויבן פון זומפּ אַ וואָרצל ווי אַ נאָז פון אַ העלפאַנד.</span>  
NLLB: On entendit un cri d'un coulonneur qui relevait de la soupe une racine comme un nez d'un serpent.

SRC: <span dir="rtl">אויף מאָרגן, אין דולער בלאָנדזשעניש, האָב איך ענדלעך דערטאַפּט דאָס בערגל. נאָר פון מיידל איז דאָרט פאַרבליבן נאָך ווינציקער ווי פון אַ חלום.</span>  
NLLB: Le lendemain matin, dans une blonde douce, j'ai finalement atteint la montagne. Seule une fille y est restée pendant vingt ans, comme dans un rêve.

SRC: <span dir="rtl">בײַם ווערן אַ פּעלצל ביסטו פאַרבליבן אַ שעפעלע.</span>  
NLLB: En devenant une pelle, vous restez une épaule.

SRC: <span dir="rtl">אַ גײַסט בין איך געווען אין דײַנע פליגל. פון זיך אַליין פאַרשוווּנדן. זיבן שטראַליקע טויבן האָבן מיך געבראַכט אַהיים.</span>  
NLLB: J'étais un hôte dans tes ailes. Je me suis disparu. Sept étoiles brillantes m'ont ramené à la maison.

SRC: <span dir="rtl">ווען כ'בין געוואָרן פאַר דיר צו גרויס, האָסטו מיר אָפּגעגעבן דײַנע הענט — די צוויי אַרבל, צו פאַרלענגערן די פרײַנדשאַפט.</span>  
NLLB: Quand je suis devenu trop grand pour toi, tu m'as donné tes mains sur les deux arbres, pour prolonger l'amitié.

SRC: <span dir="rtl">צו זײַנע אויערן דערטראָגט זיך אַ מממו, אַ גרויליק בענקענדיקער מממו, אַ דונער מיט אָפּגעהאַקטע פליגל...</span>  
NLLB: À ses oreilles se présente une mamme, une mamme grossière, une donnée aux ailes démembrées...

SRC: <span dir="rtl">— ניין, ברודערקע, אַזאַ בוים דערקענט מען בײַ נאַכט. אַרום אים, זעסט, — אַ גאָלדענער שלייער.</span>  
NLLB: Non, ma sœur, on reconnaît un tel arbre la nuit. autour de lui, voici une couverture d'or.

SRC: <span dir="rtl">— ער וועט קלעקן אָפּצובראָטן אַ סטאַדע אָקסן, טרוקן ווי פּעך, — זעצט יאָאַכים די האַק אין בוים אַרײַן.</span>  
NLLB: Il frappera et enlèvera une stade d'oiseaux, secs comme du chaume, et le jaune ramasse le jaune dans le bois.

SRC: <span dir="rtl">אַ גראָע צעכוואַליעטע וואַנט, ווי אין דער וואַנט וואָלטן זיך געטרונקען וועלף, שווימט אַרויס הינטער די זײַלן פון רעגן.</span>  
NLLB: Un mur gris et déchiré, comme dans le mur, comme des loups qui se noyeraient, nage dehors derrière les branches de la pluie.

SRC: <span dir="rtl">— אורשוליאַ, וויפל איז דער זייגער ?</span>  
NLLB: Urchuria, quelle heure est-elle ?

SRC: <span dir="rtl">דער רעגן טוט אַ זעץ אין שויב מיט זײַנע בלײַענע פויסטן.</span>  
NLLB: La pluie fait une sieste en poudre avec ses poils bleus.

SRC: <span dir="rtl">זינט איך האָב געפונען דעם ייִנגל אויפן דעמב, איז ער געוואָרן מײַן שוואַרצאַפּל.</span>  
NLLB: Depuis que j'ai trouvé le garçon sur le toit, il est devenu mon chapeau.

SRC: <span dir="rtl">און דער מענטש האָט אַרײַנגעקוקט אינעווייניק און גאָרנישט ניט געזען, ווײַל זײַן פּנים איז באַקלעפּט געוואָרן פון שפּינוועבס בײַם בייגן זיך איבערן מויל פון ברונעם.</span>  
NLLB: L'homme entre dans la maison, il regarde et il ne voit rien. Il a les yeux pleins de tristesse et il est plein de tristesse.

SRC: <span dir="rtl">און דעמאָלט האָט ער פון בוזעם אויסגעפּלאָנטערט אַ שטריק, זי פאַרקניפּט אַרום אַ האָק, און ווי אַ קוימענקערער אין אַ קוימען זיך אַרונטערגעלאָזן אין ברונעם.</span>  
NLLB: Et puis il a planté de la boue un plan, elle s'est enroulée autour d'un crochet, et comme un marchand dans un couloir se tourne autour dans le brun.

SRC: <span dir="rtl">ערשט דאָרטן, מיט אַ קלאַפּנדיק האַרץ, און מיט אויגן ווי ווילדע מאָנבלומען, האָט ער אין ביידע הענט אַרײַנגענומען דעם אוצר און עס געהאַלטן אַנטקעגן.</span>  
NLLB: D'abord là, avec un cœur battant et des yeux comme des monneaux sauvages, il prit le trésor dans ses deux mains et le déposa.

SRC: <span dir="rtl">און ער האָט נאָך אַ מאָל אין ביידע הענט אַרײַנגענומען דעם שאַרבן און ווי אַ הונט אַ צעשלאָגענער דערלאַנגט אַ וואָיע :</span>  
NLLB: Il prend de nouveau le morceau de bois dans ses deux mains. Un chien qui ne mange pas se met à crier.

SRC: <span dir="rtl">און ער האָט דערפילט, אַז דער קאָפּ, וואָס ער טראָגט שוין אַזוי פיל יאָרן אויף די אַקסלען, איז ניט זײַנער.</span>  
NLLB: Et il se rendit compte que la tête qu'il portait sur les escaliers pendant tant d'années n'était pas la sienne.

SRC: <span dir="rtl">דער מלאך מיט די פאַרסאַזשעטע פליגל טראָגט אַוועק דאָס געסל — דעם שוואַרצן אָרון.</span>  
NLLB: L'ange aux ailes déployées porte le bouclier sur le cheval noir.

SRC: <span dir="rtl">— דיר גלוסט זיך איבערשנײַדן, האַ ?</span>  
NLLB: Tu te fais du mal, hein ?

SRC: <span dir="rtl">אַצינד זעט ער די לבנה מיט זײַן גלעזערן אויג. אויף דעם אַנדערן הענגט אַ שלאָס. דער זילבערנער שליסל איז ניט מער אויף דער וועלט.</span>  
NLLB: Il voit la lune avec son œil cristallin, sur l'autre une serrure. La clé d'argent n'est plus sur la terre.

## Scores obtenus avec Torch
NLLB-200 (600M)           **sacreBLEU = 13.2**

### Alternative / complément : COMET

sacreBLEU reste une mesure de surface. Si un modèle COMET couvre votre langue, ajoutez un score sémantique — surtout utile ici, où votre système statistique produira sans doute des phrases agrammaticales que BLEU seul ne décrit pas bien.

In [28]:
try:
    from comet import download_model, load_from_checkpoint
    model_path = download_model("Unbabel/wmt22-comet-da")
    comet_model = load_from_checkpoint(model_path)
    data = [{"src": s, "mt": h, "ref": r} for s, h, r in zip(test_src, system_hyps, test_ref)]
    out = comet_model.predict(data, batch_size=8, gpus=0)
    print("COMET (IBM Model 1 + canal bruité) :", out.system_score)
except Exception as e:
    print("COMET indisponible :", e)

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1867.29it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/home/cam/git/pluriTAL/traduction-automatique/TP2/venv/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:216: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
/home/cam/git/pluriTAL/traduction-automatique/TP2/venv/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment trac

COMET (IBM Model 1 + canal bruité) : 0.3742616186539332


## 11. Bilan — questions pour votre compte-rendu

1. Comparez les trois scores sacreBLEU obtenus (mot-à-mot, IBM Model 1 + canal bruité, NLLB/LLM).
2. Donnez 3 exemples de phrases de votre corpus où votre système échoue, et reliez chaque échec à une hypothèse précise de IBM Model 1 (ordre des mots ? fertilité ? mot rare ?).
3. Le lexique bilingue extrait en section 6 est-il, lui, réellement utilisable ? Pour quel usage (pas forcément la traduction automatique complète) ?
4. Si vous deviez améliorer ce système sans passer au neuronal, quelle serait votre première piste (plus de données ? meilleur décodeur) ?
5. **Pour aller plus loin (optionnel) :** réentraînez avec plus d'itérations, ou sur un sous-ensemble plus grand de votre corpus — le score sacreBLEU de votre système progresse-t-il, et jusqu'où ?

## Réponses

### Question 1 — Comparaison des trois scores

| Système | sacreBLEU | COMET |
|---|---|---|
| Mot-à-mot | 2,8 | non calculé |
| IBM Model 1 + canal bruité | 1,3 | 0,37 |
| NLLB-200 | 13,4 | non calculé |

NLLB est loin devant (13,4), alors que les deux autres systèmes restent sous 3. Néanmoins, un score de 13 reste modeste, et cela se voit dans les sorties du modèles, du moins en français. En effet, si les traductions de NLLB sont fluides, elles contiennent parfois des erreurs (p. ex. « une stade d'oiseaux »).

De manière inattendue, **le canal bruité fait moins bien que le mot-à-mot** (1,3 contre 2,8), alors même que le modèle de langue devait améliorer les choix. Le score **COMET** (0,37) confirme que les sorties que nous avons obtenues sont très éloignées d'une traduction utilisable. Il ne faudrait pas oublier que IBM Model 1 représentait l'état de l'art en 1993 et que la traduction automatique a fait des bonds de géant depuis.

Il convient de préciser que la taille du jeu de test est réduite (30 phrases) et quelques phrases erronées ont un fort impact sur le score global.

### Question 2 — Trois échecs et l'hypothèse de IBM Model 1 correspondante

**a) Mots rares ou absents de l'entraînement**

- SRC : *<span dir="rtl">זיבן שטראַליקע טויבן האָבן מיך געבראַכט אַהיים</span>*
- HYP : « six “ voraces pupilles mansarde ” … . »
- REF : « Sept colombes nimbées de rayons m'ont ramené à la maison. »

*Zibn* → « six » est presque juste. En revanche, *shtralike tovbn* (« colombes nimbées ») donne « “ voraces », pour deux raisons différentes. *Shtralike* n'apparaît jamais à l'entraînement : t(f|e) = 0 pour tous les mots français, le tri ne départage plus que par l'ordre des mots français et retient les derniers de l'ordre Unicode (« … », « ” », « “ »). C'est ce qui explique les guillemets et les points de suspension dans beaucoup de nos sorties. *Tovbn*, en revanche, est vu 3 fois, mais plusieurs candidats sont à égalité (environ 0,06 chacun, dont « voraces ») : le choix est presque arbitraire. Le yiddish est morphologiquement riche et le corpus est petit : beaucoup de formes sont rares ou absentes.

**b) Fertilité : une locution yiddish ↔ un seul mot français**

- SRC : *<span dir="rtl">דער רעגן טוט אַ זעץ אין שויב מיט זײַנע בלײַענע פויסטן</span>*
- HYP : « éclair pluie reste un coup rouge carreau près moisissure plomb “ . »
- REF : « La pluie cogne au carreau de ses poings de plomb. »

*Tut a zets* (littéralement « fait un coup ») se traduit par le seul verbe « cogne ». IBM Model 1 n'a pas de fertilité : il ne peut pas associer trois mots source à un mot cible. Il garde « un coup » et passe à côté du verbe. Les noms isolés sont bien trouvés (« pluie », « carreau », « plomb »).

**c) Traduction libre et mots grammaticaux : l'hypothèse mot-à-mot ne tient pas**

- SRC : *<span dir="rtl">און אפשר איז גאָר שולדיק זײַן פּאַליטרע, האַ ?</span>*
- HYP : « et maternelle est … “ son palette sans hein ? »
- REF : « Et si la coupable était sa palette ? »

La référence est une traduction littéraire : *efsher* (« peut-être ») devient « si » et « hein » disparaît. Model 1 suppose qu'à chaque mot source correspond un mot cible, sans tenir compte de l'ordre ni de la construction de la phrase. Les mots rares ou inconnus sont en outre mal rendus : *efsher*, vu seulement 2 fois, a trois candidats à égalité (« seriné », « plainte », « maternelle », environ 0,13 chacun), d'où « maternelle » ; *gor* et *shuldik* n'apparaissent jamais à l'entraînement et donnent « … » et « “ » ; la virgule devient « sans ».

Ce qui fonctionne : les noms propres (« ursula », « joachim », « poznanski », « tsalke »), bien alignés aussi dans le lexique (*ursula → <span dir="rtl">אורשוליאַ</span>*, 0,99).

### Question 3 — Le lexique bilingue (section 6) est-il utilisable ?

Oui, pour un usage de glossaire, à condition de fixer un seuil de fiabilité. Cette réponse s'appuie sur le lexique lui-même (probabilités calculées par le code) et sur la validation par un LLM juge, Claude Sonnet 5.5, qui est la seule source des jugements sur le sens du yiddish (voir le complément ci-dessous : deux passes, 22 paires puis 14 paires, avec 4 contrôles chacune).

- **Mots fréquents : probabilités élevées, paires jugées correctes.** *lune → <span dir="rtl">לבנה</span>* (0,97), *rue → <span dir="rtl">גאַס</span>* (0,98), *pluie → <span dir="rtl">רעגן</span>* (0,97), *larmes → <span dir="rtl">טרערן</span>* (0,96), *ici → <span dir="rtl">דאָ</span>* (0,98). Le juge valide ces 5 paires avec une confiance haute.
- **Noms propres : probabilités élevées, paires jugées correctes.** *joachim → <span dir="rtl">יאָאַכים</span>* (0,98), *ursula → <span dir="rtl">אורשוליאַ</span>* (0,99). Le juge valide ces 2 paires avec une confiance haute.
- **La probabilité est un bon indicateur de fiabilité.** Le juge valide 5 paires sur 6 de probabilité haute (≥ 0,9, la sixième étant « partiellement correcte »), 4 sur 6 de probabilité moyenne (0,5 à 0,9) et aucune sur 6 de probabilité faible (< 0,5). Il rejette les 4 paires fausses introduites comme contrôles.
- **Mots rares : peu fiables**, car un mot français vu une ou deux fois peut recevoir une probabilité élevée avec très peu d'exemples : *heure → <span dir="rtl">זאָל</span>* (0,61), *fer → <span dir="rtl">ס'האָט</span>* (0,72), *dédaigné → <span dir="rtl">בית-עלמין</span>* (0,85) portent sur des mots français vus 1 à 3 fois à l'entraînement. Le juge les déclare **incorrectes** toutes les trois, avec une confiance haute. Dans le premier échantillon du juge, 6 des 9 paires du lexique non correctes concernent un mot français vu une seule fois, et 2 un mot vu deux fois.
- **Formes isolées, sans lemmatisation** (*larme* et *larmes* sont séparés).
- **Sens de lecture.** Le lexique est lu du français vers le yiddish : pour chaque mot français *e*, on prend le mot yiddish *f* qui maximise t(f|e). Le décodeur fait l'inverse (pour un mot yiddish *f*, il cherche le *e* qui maximise t(f|e)). Les conclusions ci-dessus portent sur le lexique, pas sur ce que produit le décodeur.

**Seuil proposé.** Sur les deux passes du juge, aucune des 13 paires de probabilité ≥ 0,9 n'est jugée incorrecte (12 correctes et 1 partiellement correcte). Parmi les 15 paires de probabilité inférieure à 0,9, 10 sont jugées incorrectes et 4 correctes : retenir les paires de probabilité ≥ 0,9 écarte donc toutes les paires jugées incorrectes, au prix de perdre 4 paires correctes. C'est une hypothèse à confirmer sur un échantillon plus grand : seule la première passe est un tirage aléatoire, la seconde porte sur des paires choisies à l'avance, et le juge est un LLM dont les verdicts n'ont pas été relus par une personne lisant le yiddish.

Usages possibles (autres que la traduction complète) : constituer un glossaire à faire relire (en gardant les mots fréquents aux probabilités élevées), repérer les noms propres et leur translittération, aider un traducteur humain à retrouver un terme, vérifier la qualité de l'alignement des phrases, amorcer une ressource pour une langue peu dotée comme le yiddish.

### Question 3 (complément) — Validation du lexique par un LLM juge (Claude Sonnet 5.5)

**Pourquoi.** Juger la justesse des paires du lexique suppose de lire le yiddish. Cette vérification ne repose donc pas sur une lecture humaine : elle est confiée à un LLM utilisé comme juge, ce qui est une pratique courante en évaluation de traduction. Elle est présentée ici avec ses limites.

**Juge (références).**
- Modèle : `claude-sonnet-5-5` (Claude Sonnet 5.5, Anthropic), identifiant indiqué par le modèle lui-même et conforme à celui de la session.
- Mise en œuvre : instance de modèle distincte, lancée comme sous-agent depuis Claude Code, le 27/10/2026. Elle n'avait **aucun outil** (pas d'accès au notebook ni au web) et **aucun accès à la conversation** : elle n'a vu que la consigne et les 22 paires ci-dessous.
- Une seule passe, sans répétition ni vote.

**Échantillon.** Tirage fait par un script extérieur au notebook, à partir du lexique de la section 6 (pour chaque mot français, le mot yiddish qui maximise t(f|e)), avec `random.seed(0)` :
- 18 paires du lexique : 6 de probabilité haute (≥ 0,9), 6 de probabilité moyenne (0,5 à 0,9), 6 de probabilité faible (< 0,5) ;
- 4 paires **de contrôle** volontairement fausses : un mot français de l'échantillon associé au mot yiddish d'un autre mot français tiré au hasard.

Les 22 paires ont été mélangées. Le juge ne voyait ni les probabilités, ni la catégorie, ni quelles paires étaient des contrôles.

**Consigne donnée au juge (résumé).** Dire si le mot yiddish est un équivalent de dictionnaire du mot français (une différence de flexion ne rend pas la paire fausse), en s'appuyant sur le contexte et sur ses connaissances du yiddish. Verdicts : *correcte*, *partiellement correcte* (sens proche, hyperonyme, autre forme grammaticale ou mot cité), *incorrecte*, *indéterminable*. Il devait ajouter une confiance (haute, moyenne, faible) et une justification d'une phrase, sans valider par politesse. Pour chaque paire, on lui fournissait au plus deux phrases du corpus d'entraînement contenant le mot yiddish, avec leur traduction française humaine (de préférence des phrases contenant aussi le mot français).

**Résultats.**

| Catégorie | Paires | correcte | partiellement correcte | incorrecte |
|---|---|---|---|---|
| Lexique, probabilité haute (≥ 0,9) | 6 | 5 | 1 | 0 |
| Lexique, probabilité moyenne (0,5 à 0,9) | 6 | 4 | 0 | 2 |
| Lexique, probabilité faible (< 0,5) | 6 | 0 | 1 | 5 |
| Contrôles (paires fausses) | 4 | 0 | 0 | 4 |

**Détail des verdicts** (verdicts et justifications reproduits tels que rendus par le juge ; la probabilité est t(f|e)) :

| # | Mot français | Mot yiddish | Type | t(f\|e) | Verdict (confiance) | Justification du juge |
|---|---|---|---|---|---|---|
| 13 | vitre | <span dir="rtl">שויב</span> | lexique, probabilité haute | 0,97 | correcte (haute) | <span dir="rtl">שויב</span> signifie « vitre, carreau » en yiddish, ce que confirment les deux phrases. |
| 22 | main | <span dir="rtl">האַנט</span> | lexique, probabilité haute | 0,96 | correcte (haute) | <span dir="rtl">האַנט</span> signifie « main » en yiddish, ce que confirment les deux phrases. |
| 1 | chêne | <span dir="rtl">דעמב</span> | lexique, probabilité haute | 0,95 | correcte (haute) | <span dir="rtl">דעמב</span> (demb) signifie « chêne » en yiddish, et le contexte confirme la traduction « chêne ». |
| 9 | souffle | <span dir="rtl">אָטעם</span> | lexique, probabilité haute | 0,95 | correcte (haute) | <span dir="rtl">אָטעם</span> signifie « souffle, haleine, respiration » en yiddish, ce que confirment les deux phrases. |
| 7 | kto | <span dir="rtl">קטאָ</span> | lexique, probabilité haute | 0,93 | partiellement correcte (moyenne) | <span dir="rtl">קטאָ</span> est la transcription du russe/polonais « kto » (qui) utilisée comme citation dans le texte ; c'est la même forme mais un mot étranger cité, pas un mot yiddish proprement dit. |
| 12 | ruelle | <span dir="rtl">געסל</span> | lexique, probabilité haute | 0,92 | correcte (haute) | <span dir="rtl">געסל</span> est le diminutif de <span dir="rtl">גאַס</span> (rue) et signifie « ruelle, petite rue », ce que confirment les deux phrases. |
| 18 | âme | <span dir="rtl">נשמה</span> | lexique, probabilité moyenne | 0,84 | correcte (haute) | <span dir="rtl">נשמה</span> signifie « âme » (hébreu <span dir="rtl">נשמה</span>), ce que confirment les deux phrases. |
| 14 | nom | <span dir="rtl">נאָמען</span> | lexique, probabilité moyenne | 0,66 | correcte (haute) | <span dir="rtl">נאָמען</span> signifie « nom », sans ambiguïté. |
| 5 | immense | <span dir="rtl">ריזיקע</span> | lexique, probabilité moyenne | 0,65 | correcte (haute) | <span dir="rtl">ריזיקע</span> est la forme fléchie de <span dir="rtl">ריזיק</span> « gigantesque, immense », qui traduit bien « immense ». |
| 6 | fleur | <span dir="rtl">אַרום</span> | lexique, probabilité moyenne | 0,64 | incorrecte (haute) | <span dir="rtl">אַרום</span> signifie « autour de » (préposition/adverbe) ; « fleur » se dit <span dir="rtl">בלום</span> ou <span dir="rtl">בלימעלע</span>, qui est le mot voisin dans la phrase. |
| 19 | poussière | <span dir="rtl">קלאַנגען</span> | lexique, probabilité moyenne | 0,55 | incorrecte (haute) | <span dir="rtl">קלאַנגען</span> signifie « sons, notes » ; « poussière » se dit <span dir="rtl">שטויב</span>, qui apparaît dans les phrases (<span dir="rtl">שטויבזײַלן, שטויב</span>). |
| 21 | danser | <span dir="rtl">טאַנצן</span> | lexique, probabilité moyenne | 0,52 | correcte (haute) | <span dir="rtl">טאַנצן</span> est l'infinitif « danser » (substantif <span dir="rtl">טאַנץ</span> = danse), confirmé par la phrase. |
| 2 | vois | <span dir="rtl">זעסטו</span> | lexique, probabilité faible | 0,37 | partiellement correcte (haute) | <span dir="rtl">זעסטו</span> est la forme verbale « zest-u » (tu vois) avec le pronom clitique « tu » accolé ; le sens est correct mais ce n'est pas le verbe seul (<span dir="rtl">זען</span> / <span dir="rtl">זעסט</span>). |
| 11 | combien | <span dir="rtl">ליגן</span> | lexique, probabilité faible | 0,25 | incorrecte (haute) | <span dir="rtl">ליגן</span> signifie « être couché, gésir » ; « combien » correspond à <span dir="rtl">װיפֿל</span>, et <span dir="rtl">ליגן</span> n'apparaît que dans la traduction « gisant ». |
| 8 | lancinant | <span dir="rtl">שפּילט</span> | lexique, probabilité faible | 0,13 | incorrecte (haute) | <span dir="rtl">שפּילט</span> est la forme verbale « il joue » ; « lancinant » correspond ici à <span dir="rtl">נאָגנדיק</span> (nagnd'ik, qui ronge), pas à <span dir="rtl">שפּילט</span>. |
| 20 | délicats | <span dir="rtl">רעכטע</span> | lexique, probabilité faible | 0,08 | incorrecte (haute) | <span dir="rtl">רעכטע</span> signifie « droit(e) » (côté droit) ; « délicats » correspond à <span dir="rtl">איידעלע</span> (adjectif fin, délicat). |
| 16 | militaire | <span dir="rtl">אַמאָל</span> | lexique, probabilité faible | 0,07 | incorrecte (haute) | <span dir="rtl">אַמאָל</span> signifie « autrefois, un jour, parfois » ; « militaire » correspond à <span dir="rtl">סאָלדאַטסקן</span> (de soldat) dans la phrase. |
| 17 | déluge | <span dir="rtl">קעסל</span> | lexique, probabilité faible | 0,06 | incorrecte (haute) | <span dir="rtl">קעסל</span> signifie « chaudron, marmite » ; « déluge » correspond à <span dir="rtl">מבול</span> (verbe <span dir="rtl">מבולט</span>), pas à <span dir="rtl">קעסל</span>. |
| 10 | main | <span dir="rtl">איז</span> | contrôle (paire fausse volontaire) | 0,00 | incorrecte (haute) | <span dir="rtl">איז</span> est le verbe « est » (être) ; « main » se dit <span dir="rtl">האַנט</span>, qui apparaît à côté dans la phrase. |
| 3 | âme | <span dir="rtl">דעם</span> | contrôle (paire fausse volontaire) | 0,00 | incorrecte (haute) | <span dir="rtl">דעם</span> est l'article défini à l'accusatif/datif masculin (« le/au »), et « âme » se dit <span dir="rtl">נשמה</span> ; ici <span dir="rtl">דעם</span> précède seulement « <span dir="rtl">בר-מיננס</span> ». |
| 4 | combien | <span dir="rtl">ראָם</span> | contrôle (paire fausse volontaire) | 0,00 | incorrecte (haute) | <span dir="rtl">ראָם</span> est le nom propre Romm (imprimerie Veuve & Frères Romm) et n'a aucun rapport avec « combien » (qui se dit <span dir="rtl">װיפֿל</span>). |
| 15 | déluge | <span dir="rtl">איבעריק</span> | contrôle (paire fausse volontaire) | 0,00 | incorrecte (haute) | <span dir="rtl">איבעריק</span> signifie « superflu, de trop » ; « déluge » se dit <span dir="rtl">מבול</span>, et l'exemple fourni ne contient pas de déluge (décalage d'alignement). |

**Lecture.**
- Les quatre contrôles sont rejetés : le juge ne valide pas n'importe quelle paire.
- Plus la probabilité est élevée, plus le juge valide : 5 paires correctes sur 6 pour une probabilité haute, 4 sur 6 pour une probabilité moyenne, aucune sur 6 pour une probabilité faible. La probabilité du lexique est donc un indicateur de fiabilité raisonnable, ce qui va dans le sens de la réponse précédente (« mots rares : peu fiables »).
- Parmi les 9 paires du lexique jugées non correctes (ou seulement partiellement), 6 concernent un mot français vu une seule fois à l'entraînement, 2 un mot vu deux fois et 1 (*vois*) un mot vu cinq fois. Les 9 paires jugées correctes concernent des mots français vus de 2 à 18 fois. Cela concorde avec l'explication donnée plus haut (un mot rare reçoit une forte probabilité par hasard), sans la démontrer sur un échantillon aussi petit.
- Le seul cas « partiellement correct » à probabilité haute (*kto*, 0,93) est un mot russe cité dans le texte et non un mot yiddish.

**Limites.**
- Le juge est un LLM : il peut se tromper en yiddish, langue peu dotée, et ses verdicts n'ont pas été relus par une personne lisant le yiddish.
- Le modèle juge est de la même famille que l'assistant qui a aidé à rédiger ce notebook : l'indépendance n'est pas totale.
- Le juge s'appuie en partie sur les traductions françaises du corpus fournies en contexte : il ne vérifie pas le lexique à partir d'une ressource extérieure.
- L'échantillon est petit (18 paires du lexique, 4 contrôles, une seule passe) : les proportions sont indicatives et non des estimations précises.
- Les trois exemples *heure*, *fer* et *dédaigné* cités dans la réponse n'étaient pas dans ce premier échantillon : ils sont jugés dans la deuxième passe ci-dessous.

#### Deuxième passe — les 10 paires citées dans la réponse

Pour vérifier les paires effectivement citées dans la réponse à la question 3, une deuxième passe a été faite avec **le même protocole** : mêmes références (`claude-sonnet-5-5`, 27/10/2026), même consigne, même format, mais une **nouvelle instance** de sous-agent, sans outil ni contexte, ne voyant ni les probabilités ni quelles paires étaient des contrôles. Elle porte sur 14 paires, mélangées : les 10 paires citées (*lune*, *rue*, *pluie*, *larmes*, *ici*, *joachim*, *ursula*, *heure*, *fer*, *dédaigné*) et 4 paires de contrôle volontairement fausses (tirage avec `random.seed(1)` : un mot français parmi les 10 associé au mot yiddish d'un autre mot français tiré au hasard).

| Catégorie | Paires | correcte | partiellement correcte | incorrecte |
|---|---|---|---|---|
| Paires citées, mots fréquents et noms propres (lune, rue, pluie, larmes, ici, joachim, ursula) | 7 | 7 | 0 | 0 |
| Paires citées, mots rares (heure, fer, dédaigné) | 3 | 0 | 0 | 3 |
| Contrôles (paires fausses) | 4 | 0 | 0 | 4 |
  
***  
  
| # | Mot français | Mot yiddish | Type | t(f\|e) | Occurrences du mot français | Verdict (confiance) | Justification du juge |
|---|---|---|---|---|---|---|---|
| 9 | lune | <span dir="rtl">לבנה</span> | paire citée dans la réponse | 0,97 | 18 | correcte (haute) | <span dir="rtl">לבנה</span> (levone) signifie « lune », et les deux phrases la traduisent par « lune ». |
| 11 | rue | <span dir="rtl">גאַס</span> | paire citée dans la réponse | 0,98 | 10 | correcte (haute) | <span dir="rtl">גאַס</span> signifie « rue », comme le montrent « de l'autre côté de la rue » et « par-dessus la rue noire ». |
| 1 | pluie | <span dir="rtl">רעגן</span> | paire citée dans la réponse | 0,97 | 14 | correcte (haute) | <span dir="rtl">רעגן</span> signifie « pluie » (regn), et les deux phrases le confirment. |
| 12 | larmes | <span dir="rtl">טרערן</span> | paire citée dans la réponse | 0,96 | 10 | correcte (haute) | <span dir="rtl">טרערן</span> est le pluriel de <span dir="rtl">טרער</span>, « larmes », confirmé par « deux larmes jumelles » et « ses larmes brûlantes ». |
| 6 | ici | <span dir="rtl">דאָ</span> | paire citée dans la réponse | 0,98 | 18 | correcte (haute) | <span dir="rtl">דאָ</span> est l'adverbe de lieu « ici », et les deux phrases le traduisent ainsi. |
| 2 | joachim | <span dir="rtl">יאָאַכים</span> | paire citée dans la réponse | 0,98 | 9 | correcte (haute) | <span dir="rtl">יאָאַכים</span> est la translittération yiddish du prénom Joachim, et la traduction française de chaque phrase donne bien « Joachim ». |
| 14 | ursula | <span dir="rtl">אורשוליאַ</span> | paire citée dans la réponse | 0,99 | 12 | correcte (haute) | <span dir="rtl">אורשוליאַ</span> est la translittération yiddish du prénom Ursula, et la traduction française de chaque phrase donne bien « Ursula ». |
| 10 | heure | <span dir="rtl">זאָל</span> | paire citée dans la réponse | 0,61 | 3 | incorrecte (haute) | <span dir="rtl">זאָל</span> est l'auxiliaire du subjonctif ou de l'optatif (« que... », « devoir »), alors que « heure » se dit <span dir="rtl">שעה</span> ou <span dir="rtl">זייגער</span> selon le sens. |
| 5 | fer | <span dir="rtl">ס'האָט</span> | paire citée dans la réponse | 0,72 | 2 | incorrecte (haute) | <span dir="rtl">ס'האָט</span> est la contraction de « es hot » (« il/cela a »), un mot grammatical sans rapport avec « fer » (en yiddish, eyzn ou <span dir="rtl">ברזל</span>). |
| 4 | dédaigné | <span dir="rtl">בית-עלמין</span> | paire citée dans la réponse | 0,85 | 1 | incorrecte (haute) | <span dir="rtl">בית-עלמין</span> signifie « cimetière », alors que « dédaigné » correspond à <span dir="rtl">פאַרשעמט</span> dans la même phrase. |
| 3 | pluie | <span dir="rtl">שלאַכט</span> | contrôle (paire fausse volontaire) | 0,00 | 14 | incorrecte (haute) | <span dir="rtl">שלאַכט</span> signifie « bataille » (la traduction française dit « cette bataille »), pas « pluie ». |
| 7 | rue | <span dir="rtl">וואָס</span> | contrôle (paire fausse volontaire) | 0,00 | 10 | incorrecte (haute) | <span dir="rtl">וואָס</span> est le pronom relatif ou interrogatif « que/qui/quoi », tandis que « rue » se dit <span dir="rtl">גאַס</span>. |
| 8 | lune | <span dir="rtl">זענט</span> | contrôle (paire fausse volontaire) | 0,00 | 18 | incorrecte (haute) | <span dir="rtl">זענט</span> est la forme « vous êtes » du verbe <span dir="rtl">זײַן</span>, sans rapport avec « lune » (<span dir="rtl">לבנה</span>). |
| 13 | ici | <span dir="rtl">קוקט</span> | contrôle (paire fausse volontaire) | 0,00 | 18 | incorrecte (haute) | <span dir="rtl">קוקט</span> est la forme verbale « regarde » (de <span dir="rtl">קוקן</span>), alors que « ici » correspond à <span dir="rtl">דאָ</span> dans la phrase, juste à côté. |

**Lecture** (verdicts du juge, non relus par une personne lisant le yiddish).
- Les 4 contrôles sont de nouveau rejetés.
- Les 7 paires de mots fréquents et de noms propres citées dans la réponse sont toutes jugées correctes, avec une confiance haute.
- Les 3 paires de mots rares (*heure*, *fer*, *dédaigné*) sont toutes jugées incorrectes, avec une confiance haute. Selon le juge, le mot yiddish proposé pour *dédaigné* correspond à « cimetière », celui proposé pour *fer* est un mot grammatical, et celui proposé pour *heure* est un auxiliaire. Dans les trois cas, le mot français n'apparaît que 1 à 3 fois dans le corpus d'entraînement.

**Limites de cette deuxième passe.** Les 10 paires ne sont pas tirées au hasard : ce sont celles que la réponse cite, donc choisies à l'avance. Elles ne remplacent pas le tirage aléatoire de la première passe pour estimer une proportion de paires correctes. Les autres limites déjà indiquées s'appliquent (juge LLM, de la même famille que l'assistant utilisé pour ce notebook, une seule passe, verdicts non relus par une personne lisant le yiddish).

### Question 4 — Première piste d'amélioration sans passer au neuronal

- 1. **Le décodeur**  

Notre décodeur tire 200 combinaisons au hasard : le canal bruité n'a donc aucune garantie de faire au moins aussi bien que le mot-à-mot. Une recherche en faisceau (*beam search*), ou au minimum l'ajout systématique de la combinaison gloutonne parmi les candidats, est peu coûteuse et vérifiable rapidement.

- 2. **Plus de données.**  

430 paires d'entraînement sont très peu : IBM Model 1 s'entraîne normalement sur des dizaines de milliers de phrases, et une grande partie du vocabulaire de test est inconnue. Un corpus plus grand réduirait les mots inconnus, cause majeure des échecs (question 2a).

Autres pistes, plus coûteuses : un modèle plus riche (IBM 2 pour l'ordre des mots, IBM 3 pour la fertilité), un modèle de langue trigramme avec lissage, une lemmatisation côté français.

### Question 5 (optionnelle) — Plus d'itérations ou plus de données

sacreBLEU sur les 30 phrases de test, en ne changeant que `N_ITER`.  
- Mot-à-mot : déterministe
- Canal bruité (décodeur aléatoire) : moyenne de 5 tirages et (min – max).  


- N_ITER = 1 -> mot-à-mot : 0,1 | canal bruité : 0,1 (0,1 – 0,1)
- N_ITER = 2 -> mot-à-mot : 1,0 | canal bruité : 1,1 (0,8 – 1,4)
- N_ITER = 5 -> mot-à-mot : 2,5 | canal bruité : 1,9 (1,7 – 2,6)
- N_ITER = 10 -> mot-à-mot : 2,8 | canal bruité : 1,9 (1,0 – 2,7)
- N_ITER = 20 -> mot-à-mot : 2,8 | canal bruité : 1,8 (0,7 – 2,8)
- N_ITER = 50 -> mot-à-mot : 2,5 | canal bruité : 1,6 (1,3 – 1,8)

**Conclusion.** Le score progresse très vite au début, puis plafonne : on passe de 0,1 avec une seule itération à 2,5 avec 5 itérations (mot-à-mot), puis à 2,8 avec 10 ou 20 itérations. Au-delà, plus d'itérations n'apportent rien : à 50 itérations, le score est même un peu plus bas (2,5 pour le mot-à-mot, 1,6 pour le canal bruité). Pour le canal bruité, l'écart entre 5, 10 et 20 itérations (1,9 ; 1,9 ; 1,8) est plus petit que la variation d'un tirage à l'autre du décodeur (par exemple de 0,7 à 2,8 à 20 itérations) : on ne peut donc pas dire que l'un de ces réglages est meilleur que les autres.

Le score est limité à environ 3 par autre chose que le nombre d'itérations. Mon hypothèse, que je n'ai pas testée, est que ce sont les données : avec 430 paires d'entraînement, beaucoup de mots du jeu de test sont inconnus ou trop rares pour que le modèle apprenne leur traduction, quel que soit le nombre d'itérations. Pour le vérifier, il faudrait réentraîner sur des sous-ensembles de plus en plus grands du corpus (par exemple 100, 200 puis 430 paires) et voir si le score augmente avec la quantité de données.
